# Разведка API маркетплейса

Шаги:
1. Первый запрос и осмотр ответа
2. Парсинг JSON, изучение структуры
3. Сохранение «сырого» ответа
4. Если не 200 — подбор метода/параметров/авторизации
5. Пагинация
6. Rate limit (аккуратно)
7. Что приходит за «пустой» день
8. Самая ранняя дата с данными (бинарный поиск)

**Перед запуском:** создать `.env` рядом с ноутбуком:
```
API_URL=https://...
API_TOKEN=...
```


In [ ]:
# Импорты
import os
import json
import time
import requests
from datetime import date, timedelta
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv

load_dotenv()

RAW_DIR = Path("raw")
RAW_DIR.mkdir(exist_ok=True)

## Конфиг

In [ ]:
API_URL   = os.getenv("API_URL", "ВСТАВЬ_СЮДА_СВОЙ_URL")
API_TOKEN = os.getenv("API_TOKEN")  # если требуется

# Базовые заголовки. Тип авторизации (Bearer/X-Api-Key/Basic) подберём ниже.
HEADERS = {"Accept": "application/json"}
if API_TOKEN:
    HEADERS["Authorization"] = f"Bearer {API_TOKEN}"

TEST_DATE = (date.today() - timedelta(days=1)).isoformat()  # вчера

print("URL:", API_URL)
print("Test date:", TEST_DATE)

## Шаг 1. Первый запрос

Самый вероятный сценарий: **GET, дата в query**. Имя параметра неизвестно — пробуем кандидатов и останавливаемся на первом 200.

In [ ]:
PARAM_CANDIDATES = ["date", "day", "dt", "on_date", "from", "report_date"]

r = None
used_param = None
last_resp = None

for p in PARAM_CANDIDATES:
    resp = requests.get(API_URL, params={p: TEST_DATE}, headers=HEADERS, timeout=30)
    print(f"param={p:>12}  status={resp.status_code}  url={resp.url}")
    last_resp = resp
    if resp.status_code == 200 and r is None:
        r = resp
        used_param = p

if r is None:
    print("\n⚠️  Ни один GET-кандидат не вернул 200. Иди на Шаг 4 — подбор метода.")
    r = last_resp
    used_param = "date"  # дефолт, чтобы дальнейшие ячейки не падали
else:
    print("\n✅ Победил параметр:", used_param)

## Шаг 2. Осмотр ответа

In [ ]:
# Заголовки — подсказки про rate limit, пагинацию, тип контента
print("--- HEADERS ---")
for k, v in r.headers.items():
    print(f"{k}: {v}")

print("\n--- BODY (первые 500 символов) ---")
print(r.text[:500])

In [ ]:
# Парсим JSON
data = None
try:
    data = r.json()
    print("Тип ответа:", type(data).__name__)
    if isinstance(data, dict):
        print("Ключи верхнего уровня:", list(data.keys()))
    elif isinstance(data, list):
        print("Длина списка:", len(data))
        if data and isinstance(data[0], dict):
            print("Ключи первого элемента:", list(data[0].keys()))
except ValueError:
    print("⚠️  Это не JSON. Проверь Content-Type выше.")

In [ ]:
# Беглый осмотр одной записи
sample = None
records_key = None

if isinstance(data, list) and data:
    sample = data[0]
elif isinstance(data, dict):
    for key in ("items", "data", "results", "rows", "orders", "result"):
        if key in data and isinstance(data[key], list) and data[key]:
            sample = data[key][0]
            records_key = key
            print(f"Записи лежат в data['{key}']")
            break
    if sample is None:
        sample = data

print()
pprint(sample, depth=3, sort_dicts=False)

## Шаг 3. Сохранить «сырой» ответ

Полный — для дальнейшей работы. Маленький (1-3 записи) — удобно показать в следующем чате при проектировании БД.

In [ ]:
# Полный ответ
out_full = RAW_DIR / f"{TEST_DATE}_full.json"
out_full.write_text(json.dumps(data, ensure_ascii=False, indent=2))
print("Полный ответ:", out_full, f"({out_full.stat().st_size} байт)")

# Мини-пример
if isinstance(data, list):
    sample_data = data[:3]
elif isinstance(data, dict):
    sample_data = dict(data)
    if records_key:
        sample_data[records_key] = sample_data[records_key][:3]
else:
    sample_data = data

out_sample = RAW_DIR / f"{TEST_DATE}_sample.json"
out_sample.write_text(json.dumps(sample_data, ensure_ascii=False, indent=2))
print("Мини-пример: ", out_sample, f"({out_sample.stat().st_size} байт)")

## Шаг 4. Если не получили 200 — подбираем

Чек-лист в порядке убывания вероятности:

1. **Другой query-параметр** — уже сделано в Шаге 1
2. **POST с датой в JSON body**
3. **Дата в path**: `{base_url}/2024-01-15`
4. **Авторизация**: `Bearer` / `X-Api-Key` / `Basic`
5. **Content-Type: application/json** на POST

Шаблоны ниже — раскомментируй нужное.

In [ ]:
# POST + JSON body
# r = requests.post(API_URL,
#                   json={"date": TEST_DATE},
#                   headers={**HEADERS, "Content-Type": "application/json"},
#                   timeout=30)
# print(r.status_code, r.text[:300])

# Дата в path
# url_path = f"{API_URL.rstrip('/')}/{TEST_DATE}"
# r = requests.get(url_path, headers=HEADERS, timeout=30)
# print(r.status_code, r.text[:300])

# X-Api-Key вместо Bearer
# r = requests.get(API_URL,
#                  params={"date": TEST_DATE},
#                  headers={"X-Api-Key": API_TOKEN, "Accept": "application/json"},
#                  timeout=30)
# print(r.status_code, r.text[:300])

# Basic-auth
# from requests.auth import HTTPBasicAuth
# r = requests.get(API_URL, params={"date": TEST_DATE},
#                  auth=HTTPBasicAuth("user", "pass"), timeout=30)
# print(r.status_code, r.text[:300])

## Шаг 5. Пагинация

Признаки:
- В JSON: `total`, `count`, `next`, `has_more`, `page`, `pages`, `offset`, `limit`, `cursor`
- В заголовках: `Link`, `X-Total-Count`, `X-Page`, `X-Per-Page`
- Косвенный: ровно 100 / 500 / 1000 записей — почти точно дефолтный лимит

In [ ]:
# В JSON
pag_in_json = []
if isinstance(data, dict):
    for k in data:
        if any(s in k.lower() for s in ("total", "count", "next", "has_more", "page", "offset", "limit", "cursor")):
            v = data[k] if not isinstance(data[k], (list, dict)) else type(data[k]).__name__
            pag_in_json.append((k, v))
print("Пагинация в JSON:    ", pag_in_json)

# В заголовках
pag_in_headers = {k: v for k, v in r.headers.items()
                  if any(s in k.lower() for s in ("link", "total", "page", "limit", "next", "cursor"))}
print("Пагинация в headers:", pag_in_headers)

# Сколько записей в ответе
n = None
if isinstance(data, list):
    n = len(data)
elif isinstance(data, dict) and records_key:
    n = len(data[records_key])
print(f"\nЗаписей в ответе: {n}  (если ровно 100/500/1000 — почти точно есть пагинация)")

In [ ]:
# Если есть пагинация — проверяем вторую страницу
# r2 = requests.get(API_URL,
#                   params={used_param: TEST_DATE, "page": 2},
#                   headers=HEADERS, timeout=30)
# print(r2.status_code)
# print(r2.text[:500])

## Шаг 6. Rate limit (аккуратно)

Сначала пассивно — смотрим заголовки, многие API сами говорят про лимиты. Если ничего нет — мягкий тест: ~5 запросов с паузой 0.5 сек, ловим 429. **Не превращай в стресс-тест.**

In [ ]:
# Пассивно
rate_headers = {k: v for k, v in r.headers.items()
                if any(s in k.lower() for s in ("rate", "limit", "retry"))}
print("Rate-headers:", rate_headers)

In [ ]:
# Активно — 5 запросов, пауза 0.5 сек
for i in range(5):
    t0 = time.time()
    rr = requests.get(API_URL, params={used_param: TEST_DATE}, headers=HEADERS, timeout=30)
    dt = time.time() - t0
    print(f"#{i+1}: status={rr.status_code}  time={dt:.2f}s")
    if rr.status_code == 429:
        print("⚠️ 429! Retry-After:", rr.headers.get("Retry-After"))
        break
    time.sleep(0.5)

## Шаг 7. «Пустой» день

Что вернётся, если данных нет — пустой массив? `count: 0`? 404? Хорошие тесты: завтра (будущее) и 2010-01-01 (до маркетплейса).

In [ ]:
for test in [(date.today() + timedelta(days=1)).isoformat(), "2010-01-01"]:
    rr = requests.get(API_URL, params={used_param: test}, headers=HEADERS, timeout=30)
    print(f"\nДата: {test}   status: {rr.status_code}")
    try:
        body = rr.json()
        if isinstance(body, dict):
            print("Ключи:", list(body.keys()))
        print("Превью:", json.dumps(body, ensure_ascii=False)[:200])
    except ValueError:
        print("Не JSON:", rr.text[:200])
    time.sleep(0.5)

## Шаг 8. Самая ранняя доступная дата (бинарный поиск)

Идея: данные доступны от какого-то X и до сегодня. Бинарный поиск по `[2015-01-01, today]`. На 10-летнем диапазоне — ~12-13 запросов.

In [ ]:
def has_data(d: date) -> bool:
    """True, если за день d пришли непустые данные.
    После Шага 7 поправь логику пустоты под формат ответа."""
    rr = requests.get(API_URL, params={used_param: d.isoformat()},
                      headers=HEADERS, timeout=30)
    if rr.status_code != 200:
        return False
    try:
        body = rr.json()
    except ValueError:
        return False
    if isinstance(body, list):
        return len(body) > 0
    if isinstance(body, dict):
        for key in ("items", "data", "results", "rows", "orders"):
            if key in body and isinstance(body[key], list) and len(body[key]) > 0:
                return True
        for key in ("total", "count"):
            if key in body and isinstance(body[key], (int, float)) and body[key] > 0:
                return True
    return False

def find_earliest_day(low=date(2015, 1, 1), high=None, pause=0.5) -> date:
    if high is None:
        high = date.today() - timedelta(days=1)
    if not has_data(high):
        raise RuntimeError("Даже у high нет данных — поправь параметры")
    while low < high:
        mid = low + (high - low) // 2
        time.sleep(pause)
        if has_data(mid):
            high = mid
        else:
            low = mid + timedelta(days=1)
        print(f"  низ={low}  верх={high}")
    return low

# earliest = find_earliest_day()
# print("Самая ранняя дата с данными:", earliest)

## Финальный чек-лист — заполнить после разведки

**Транспорт**
- [ ] Метод: GET / POST
- [ ] Где дата: query / body / path
- [ ] Имя параметра даты
- [ ] Формат даты

**Авторизация**
- [ ] Нужна / нет
- [ ] Тип: Bearer / X-Api-Key / Basic / другое

**Пагинация**
- [ ] Есть / нет
- [ ] Тип: page+limit / offset+limit / cursor / Link
- [ ] Дефолтный размер страницы
- [ ] Признак конца страниц

**Rate limit**
- [ ] Есть / нет
- [ ] Лимит на минуту/час
- [ ] Поведение при 429: Retry-After / просто ждать

**Данные**
- [ ] Самая ранняя дата с данными
- [ ] Что приходит за пустой день
- [ ] Уровень записи: одна строка = заказ / позиция / что-то ещё
- [ ] Среднее число записей в день

**Сущности и поля для БД** (отметить, что есть в ответе):

*Заказ:* order_id, дата/время, статус, сумма, способ оплаты, способ доставки, регион/склад
*Клиент:* customer_id, регион/город, дата регистрации
*Товар (SKU):* product_id, sku, название, категория, бренд, цена
*Позиция заказа:* order_id, product_id, количество, цена за шт, итого
*Скидки/промо:* тип, сумма скидки, % скидки, промокод
*Логистика/финансы:* комиссия, доставка, возврат, рефанд

---

## Что взять в следующий чат для проектирования БД

1. `raw/{дата}_sample.json` — мини-пример (1-3 записи)
2. Заполненный чек-лист выше
3. Оценка объёма — записей в день × количество дней истории